# ClaimPilot Pipeline Evaluation
End-to-end testing and metrics for the multi-agent claims processing pipeline

In [ ]:
import json
import os
import sys
import time
import asyncio
from datetime import datetime

sys.path.insert(0, '../backend')

from orchestrator.pipeline import pipeline
from orchestrator.state_schema import ClaimState

print('Imports successful')

In [ ]:
# Load mock claims for testing
claims_path = '../data/mock_claims/synthetic_claims.json'
if os.path.exists(claims_path):
    with open(claims_path, 'r') as f:
        claims = json.load(f)
    if isinstance(claims, dict) and 'records' in claims:
        claims = claims['records']
    print(f'Loaded {len(claims)} test claims')
else:
    print('No synthetic claims found - generate them first')
    claims = []

In [ ]:
# Run pipeline on a single test claim
async def run_test_claim(claim: dict) -> dict:
    import uuid

    initial_state = {
        "claim_id": str(uuid.uuid4()),
        "raw_input": {
            "type": "text",
            "content": json.dumps(claim),
        },
        "claim_payload": {},
        "validation_result": {},
        "investigation_bundle": {},
        "fraud_assessment": {},
        "settlement_decision": {},
        "reasoning_chain": [],
        "current_status": "SUBMITTED",
        "current_agent": "",
        "error_log": [],
        "retry_count": {},
        "pipeline_started_at": datetime.utcnow().isoformat(),
        "pipeline_completed_at": "",
    }

    start = time.time()
    result = await pipeline.run(initial_state)
    elapsed = time.time() - start

    return {
        "claim": claim,
        "result": result,
        "elapsed_seconds": round(elapsed, 2),
        "final_status": result.get('current_status'),
        "reasoning_steps": len(result.get('reasoning_chain', [])),
    }

if claims:
    result = await run_test_claim(claims[0])
    print(f"Test claim completed in {result['elapsed_seconds']}s")
    print(f"Final status: {result['final_status']}")
    print(f"Reasoning steps: {result['reasoning_steps']}")
else:
    print('No claims to test')

In [ ]:
# Run batch evaluation on multiple claims
async def batch_evaluate(test_claims: list, sample_size: int = 10):
    results = []
    test_batch = test_claims[:sample_size]

    for i, claim in enumerate(test_batch):
        print(f'Processing claim {i+1}/{len(test_batch)}...', end=' ')
        try:
            result = await run_test_claim(claim)
            results.append(result)
            print(f'✓ {result["final_status"]} ({result["elapsed_seconds"]}s)')
        except Exception as e:
            print(f'✗ Error: {e}')
            results.append({'error': str(e), 'claim': claim})

    return results

if claims:
    batch_results = await batch_evaluate(claims, 5)

    # Summary metrics
    successful = [r for r in batch_results if 'error' not in r]
    failed = [r for r in batch_results if 'error' in r]

    print(f'\n=== Pipeline Evaluation Summary ===')
    print(f'Total tested: {len(batch_results)}')
    print(f'Successful: {len(successful)}')
    print(f'Failed: {len(failed)}')

    if successful:
        avg_time = sum(r['elapsed_seconds'] for r in successful) / len(successful)
        print(f'Average pipeline time: {avg_time:.2f}s')

        statuses = {}
        for r in successful:
            s = r['final_status']
            statuses[s] = statuses.get(s, 0) + 1
        print(f'Status distribution: {statuses}')
else:
    print('No claims to evaluate')

In [ ]:
# Analyze reasoning chain depth
if claims:
    for r in successful[:3]:
        chain = r['result'].get('reasoning_chain', [])
        print(f"\nClaim {r['claim'].get('claim_number', 'N/A')}")
        print(f"  Final: {r['final_status']} in {r['elapsed_seconds']}s")
        print(f"  Chain length: {len(chain)} steps")
        for step in chain:
            agent = step.get('agent', '?')
            action = step.get('action', '?')
            print(f"    [{agent}] {action}")
else:
    print('No results to analyze')